# Aegis.dv – Hospital Data Cleaning & Standardization Pipeline

**Project:** Aegis.dv – Hospital Operations & Patient Analytics Dashboard  
**Author:** Ananya Biju (Infosys Springboard Internship)  
**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008) Inpatient Encounters  

This interactive notebook demonstrates the complete clinical data cleaning, standardization, and quality auditing workflow.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.preprocessing.data_loader import load_raw_data, load_id_mappings
from src.validation.schema_validator import validate_raw_schema
from src.preprocessing.deduplication import deduplicate_encounters
from src.preprocessing.missing_values import replace_placeholder_tokens, analyze_missing_values
from src.preprocessing.missing_handler import handle_missing_values
from src.preprocessing.department_standardizer import standardize_departments
from src.preprocessing.timeline_normalizer import normalize_encounter_timeline
from src.preprocessing.type_converter import convert_data_types
from src.validation.outlier_detector import detect_outliers_iqr
from src.feature_engineering.feature_builder import build_derived_features
from src.kpi.kpi_engine import compute_headline_kpis, calculate_department_summary

## 1. Data Ingestion & Schema Contract Verification

In [ ]:
raw_dir = project_root / 'data' / 'raw'
df_raw = load_raw_data(raw_dir)
print(f'Raw encounters ingested: {len(df_raw):,} rows, {len(df_raw.columns)} columns')

schema_report = validate_raw_schema(df_raw)
print('Schema Validation Status:', schema_report['status'])

## 2. Deduplication on Primary Encounter Key

In [ ]:
df_dedup, dedup_audit = deduplicate_encounters(df_raw, key_column='encounter_id')
print(f"Initial rows: {dedup_audit['initial_row_count']:,}")
print(f"Duplicates detected on encounter_id: {dedup_audit['duplicates_detected']}")
print(f"Unique patients tracked: {dedup_audit['unique_patients']:,}")

## 3. Missing Value Audit & Principled Clinical Categorization

In [ ]:
df_clean_tokens = replace_placeholder_tokens(df_dedup)
raw_missing_audit = analyze_missing_values(df_clean_tokens)
print(f"Raw completeness score: {raw_missing_audit['overall_completeness_pct']}%")

# Apply principled treatment
df_treated, handling_audit = handle_missing_values(df_clean_tokens)
print('Missing value treatments applied across attributes:')
for col, info in handling_audit.items():
    print(f"  - {col:18s}: {info['action']} (Count: {info['count_treated']:,})")

## 4. Medical Specialty to Clinical Department Standardization

In [ ]:
mapping_path = project_root / 'data' / 'interim' / 'specialty_mapping.csv'
df_dept, dept_audit = standardize_departments(df_treated, mapping_path)
print(f"Clinical departments standardized: {dept_audit['standardized_departments_count']}")
print(f"Unmapped specialties: {dept_audit['unmapped_records']}")
print("\nTop 5 Departments by Patient Volume:")
for dept, count in list(dept_audit['department_distribution'].items())[:5]:
    print(f"  - {dept:32s}: {count:,} encounters")

## 5. Timeline Normalization & Type Enforcement

In [ ]:
df_time, time_audit = normalize_encounter_timeline(df_dept)
df_typed, type_audit = convert_data_types(df_time)
print(f"Reference timeline: {time_audit['start_date']} to {time_audit['end_date']}")
print(f"Attribution label: {time_audit['timeline_label']}")

## 6. Outlier Analysis & Derived Feature Engineering

In [ ]:
df_outliers, outlier_audit = detect_outliers_iqr(df_typed)
df_final, feature_audit = build_derived_features(df_outliers)

final_missing_audit = analyze_missing_values(df_final)
print(f"Final Dataset Shape: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print(f"Final Completeness Score: {final_missing_audit['overall_completeness_pct']}% (Target > 95.0% ACHIEVED)")
print(f"Surviving Readmission Eligible Cohort: {feature_audit['total_readmission_eligible']:,} encounters")
print(f"In-Hospital Mortality Excluded: {feature_audit['total_in_hospital_deaths']:,} deaths")

## 7. Executive Operational Scorecard

In [ ]:
headline = compute_headline_kpis(df_final)
print('=== AEGIS.DV OPERATIONAL HEADLINE SCORECARD ===')
for k, v in headline.items():
    print(f"{k:30s}: {v}")